# SAM2 Auto‑Prompt → Track (Kaggle GPU)

This notebook: 
1) loads bubble frames + ROI mask,
2) searches for the first SAM2 auto‑mask prompt,
3) tracks with SAM2 video predictor,
4) writes overlay video.

**Prereqs (Kaggle):**
- Enable GPU (Settings → Accelerator → GPU)
- Add a dataset that contains: `Air1/` frames and `roi_mask.png`
- Add this repo as a dataset or clone from GitHub.


In [ ]:
import torch, os, sys, shutil, json
print('cuda available:', torch.cuda.is_available())


## Paths
Update dataset paths below.


In [ ]:
DATA_DIR = '/kaggle/input/bubbles-dataset'  # contains Air1/ frames
ROI_MASK = '/kaggle/input/bubbles-dataset/roi_mask.png'
WORK_DIR = '/kaggle/working'
REPO_DIR = '/kaggle/working/bubble_tracking'
OUT_DIR = '/kaggle/working/outputs/bubbles_air1_072_271_auto_search'


## Repo setup
If you added the repo as a Kaggle dataset, copy it into `/kaggle/working`.
Otherwise, clone it (requires Internet).


In [ ]:
if not os.path.exists(REPO_DIR):
    if os.path.exists('/kaggle/input/bubble_tracking'):
        shutil.copytree('/kaggle/input/bubble_tracking', REPO_DIR)
    else:
        !git clone <YOUR_REPO_URL> {REPO_DIR}

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))


## Install SAM2 (editable)
If `segment-anything-2` is already in the repo, install it in editable mode.


In [ ]:
%cd /kaggle/working/bubble_tracking
!pip -q install -U hydra-core iopath pyyaml opencv-python tqdm matplotlib
!pip -q install -e segment-anything-2 --no-build-isolation


## Checkpoint
Place `sam2_hiera_tiny.pt` in `checkpoints/`. If missing and Internet is enabled, download it.


In [ ]:
ckpt = os.path.join(REPO_DIR, 'checkpoints', 'sam2_hiera_tiny.pt')
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
if not os.path.exists(ckpt):
    !wget -q -O {ckpt} https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt
print('ckpt exists:', os.path.exists(ckpt))


## Build frame subset (72–271)
This recreates the numeric frame folder used by the scripts.


In [ ]:
import re
from pathlib import Path

src = Path(DATA_DIR) / 'Air1'
dst_root = Path('/kaggle/working/data/interim/bubbles_air1_072_271')
dst = dst_root / 'frames'
if dst_root.exists():
    shutil.rmtree(dst_root)
dst.mkdir(parents=True, exist_ok=True)

pat = re.compile(r'\((\d+)\)\.(jpg|jpeg|png)$', re.I)
idx_to_file = {}
for f in os.listdir(src):
    m = pat.search(f)
    if m:
        idx_to_file[int(m.group(1))] = f

start, end = 72, 271
map_lines = []
for new_idx, orig_idx in enumerate(range(start, end+1)):
    fname = idx_to_file[orig_idx]
    shutil.copy2(src / fname, dst / f'{new_idx:06d}.jpg')
    map_lines.append(f'{new_idx},{orig_idx},{fname}')

(dst_root / 'map.csv').write_text('new_idx,orig_idx,orig_name\n' + '\n'.join(map_lines))
print('frames:', len(list(dst.glob('*.jpg'))))


## Auto‑prompt then track
This searches for the first prompt via SAM2 automask, then tracks.


In [ ]:
!python scripts/run_sam2_prompt_then_track.py \
  --frames /kaggle/working/data/interim/bubbles_air1_072_271/frames \
  --output {OUT_DIR} \
  --checkpoint {ckpt} \
  --model-cfg sam2_hiera_t.yaml \
  --device cuda \
  --roi-mask {ROI_MASK} \
  --start-frame 0 \
  --max-search-frames 200 \
  --points-per-side 16 \
  --min-area 20 \
  --max-area-ratio 0.5 \
  --mask-scale 0.5 \
  --max-prompts 1 \
  --prompts-out /kaggle/working/data/interim/bubbles_air1_072_271/prompts_auto_sam2.yaml \
  --fps 10


## Output
Overlay video should be in `OUT_DIR/overlay.mp4`.
